<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP9.2B/MP9.2B_air_channel.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP9.2B · The Air Channel

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'channel.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-hydrogen-channel/")
PROJECT = (['mp92b.py', 'MP9.2B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP9.2B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex09.2-hydrogen-channel", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex09.2-hydrogen-channel", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'channel', 'mp92b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import channel as ch
import mp92b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP9.2B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Ex09.2 followed the fuel along the fuel channel of a solid oxide fuel cell (SOFC). This mini project follows the **air** along the channel on the other side of the same cell. At the air electrode oxygen is taken from the air - four electrons per oxygen molecule - and crosses the electrolyte as oxide ions. Air is fed far in excess of what the current needs, because it also carries the heat out of the stack: a **stoichiometry** (air fed over the oxygen the current takes) of about five is typical.

Two quantities are wanted: **the oxygen fraction at the electrode along the channel**, which sets the local current and says whether the electrode near the outlet starves; and **the air speed at which the electrode at the outlet still sees 15 % oxygen**, a design margin. A stack controller or a design study asks them again for every new air flow, so the model is run many times. That is where a fast model pays.

This mini project computes both for any air speed at the inlet between 1.0 and 3.0 m/s, with one network.

## A2 · The channel, and what is new against Ex09.2

The channel has Ex09.2's dimensions: 100 mm long, 1 mm high, at 800 °C and 1 atm. Three things change.

**1 · Oxygen is taken and nothing is returned.** On the fuel side one molecule of steam returns for every molecule of hydrogen taken, so the molar flow and the velocity are constant along the channel. On the air side the oxygen leaves through the wall and nothing comes back: the **molar flow falls along the channel** (the nitrogen is conserved), the gas acquires a small velocity towards the electrode (the **Stefan velocity**, $v = -N/c$ at the wall), and the diffusive flux at the wall carries only the share $1 - y_w$ of the oxygen taken, because the gas moving towards the wall carries the rest. J. Khazaei's review of Ex09.2 (7 October 2026) pointed out the first consequence: the oxygen fraction of the mixed gas at the outlet is $(y_\mathrm{in} - Y)/(1 - Y)$, not the fuel side's $y_\mathrm{in} - Y$.

**2 · Air, not hydrogen.** 21 % oxygen in nitrogen; oxygen diffuses in nitrogen about four times more slowly than steam in hydrogen, $D$ = 1.9e-04 m²/s at 800 °C (an estimate); four electrons per molecule, so half the molar flux of the fuel side at the same current.

**3 · The current follows the oxygen at the electrode.** Ex09.2 prescribed a uniform 0.5 A/cm². A cell runs at a fixed voltage, and where the oxygen at the electrode is depleted the concentration overpotential rises and the local current falls. This mini project uses a reduced model of that dependence, a saturating (Langmuir-type) form:

$$\frac{N}{N_0} = g(y_w) = (1 - B) + B\,\frac{r(y_w)}{r(y_\mathrm{in})}, \qquad r(y) = \frac{y}{y + K}$$

with $y_w$ the oxygen fraction at the electrode, $N_0 = i_0/4F$ the flux at the inlet's composition with Ex09.2's $i_0$ = 0.5 A/cm², and $K$ = 0.05. The form and $K$ are **assumed**: no kinetic constant is taken from a measurement.

Two strengths switch the changes off: $B$ (the coupling) and $S$ (the one-sided exchange). At $B = 0$ and $S = 0$ the problem is Ex09.2's with the air's numbers and the flux reversed; this project uses $B = 1$ and $S = 1$.

In [ ]:
# The air channel and its data
rows = [("channel", f"{ch.L_CHANNEL*1e3:.0f} mm long, {ch.H_CHANNEL*1e3:.0f} mm high, {ch.T_CELL-273.15:.0f} °C, 1 atm, c = {ch.C_TOT:.2f} mol/m³"),
        ("air fed", f"{100*mp.Y_IN:.0f} % oxygen; D = {mp.D_AIR:.1e} m²/s, μ = {mp.MU_AIR:.1e} Pa s, ρ = {mp.RHO_AIR:.2f} kg/m³ (estimates)"),
        ("air speed", f"u = {mp.S_MIN:.1f} to {mp.S_MAX:.1f} m/s at the inlet, a network input: stoichiometry "
                      f"{mp.stoichiometry(mp.S_MIN):.2f} to {mp.stoichiometry(mp.S_MAX):.2f}"),
        ("current", f"{ch.I_CELL/1e4:.1f} A/cm² at the inlet's composition, then following the oxygen at the electrode"),
        ("wall flux", f"N0 = i0/4F = {mp.N0:.4f} mol/(m² s) of oxygen taken, nothing returned"),
        ("coupling", f"K = {mp.K_O2:.2f} (assumed), strength B = {mp.B_COUPLE:.0f}; one-sided exchange S = {mp.STEFAN:.0f}")]
for k, v in rows:
    print(f"{k:>10}  {v}")
s5 = mp.speed_for_stoichiometry(5.0)
print(f"\nstoichiometry 5 at {s5:.2f} m/s: Re = {mp.reynolds(s5):.1f}, pressure drop {mp.pressure_drop(s5):.0f} Pa; "
      f"Stefan velocity at the wall N0/c = {mp.N0/ch.C_TOT*1e3:.2f} mm/s against D/h = {mp.D_AIR/ch.H_CHANNEL*1e3:.0f} mm/s")

**What you should see.** The air at 0.33 kg/m³, the speeds from 1.0 to 3.0 m/s spanning stoichiometries 1.84 to 5.52, and the oxygen flux, half the fuel side's hydrogen flux. At stoichiometry 5 the air runs at 2.72 m/s (Reynolds number 20, pressure drop 143 Pa). The Stefan velocity is 1.14 mm/s, 0.6 % of the diffusion velocity $D/h$: small, and part C measures what it changes.

In [ ]:
# The current density against the oxygen at the electrode
y = np.linspace(0.005, 0.21, 200)                        # oxygen fraction at the electrode
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(100 * y, mp.current_density((y - mp.Y_IN) / mp.Y), label="B = 1, this project")
ax.plot(100 * y, mp.current_density((y - mp.Y_IN) / mp.Y, b=0.0), "--", label="B = 0, uniform (Ex09.2)")
ax.set_xlabel("oxygen fraction at the electrode [%]"); ax.set_ylabel("current density [A/cm²]"); ax.legend()
plt.tight_layout(); plt.show()
for yy in (0.21, 0.15, 0.10, 0.05):
    print(f"oxygen at the electrode {100*yy:4.1f} %: current density {float(mp.current_density((yy - mp.Y_IN) / mp.Y)):.2f} A/cm²")

**What you should see.** 0.50 A/cm² with the air's 21 % oxygen, 0.46 at 15 %, 0.41 at 10 % and 0.31 at 5 %: the current falls slowly while there is oxygen and steeply once it runs short.

## A3 · The domain, the equation and its conditions

The domain is Ex09.2's: $0 < x < L$ along the channel, $0 < z < h$ across it ($z = 0$ the air electrode, $z = h$ the interconnect). The unknown is the oxygen fraction $y(x, z)$, which depends on the air speed too. The axial velocity is Poiseuille's profile with the local mean speed $\bar u(x) = u_\mathrm{in}\,\sigma(x)$, where $\sigma$ is the molar flow left, relative to the inlet's; the cross velocity follows from continuity, $v(z) = -(N/c)\,f(z/h)$ with $f(Z) = 1 - 3Z^2 + 2Z^3$ (1 at the electrode, 0 at the interconnect). The species balance in mole fractions is

$$u(x, z)\,\frac{\partial y}{\partial x} + v(z)\,\frac{\partial y}{\partial z} = D\,\frac{\partial^2 y}{\partial z^2} \quad \text{in the channel}$$

| where | condition | why it holds |
|---|---|---|
| inlet ($x = 0$) | $y = 0.21$ | the air fed |
| electrode ($z = 0$) | $D c\,\dfrac{\partial y}{\partial z} = N_0\,g(y)\,(1 - y)$ | oxygen leaves at the rate the local current takes it; diffusion carries the share $1 - y$, the Stefan flow the rest |
| interconnect ($z = h$) | $\partial y/\partial z = 0$ | an impermeable steel wall |
| the flow | $\sigma(x)\,(1 - \bar y(x)) = 1 - 0.21$ | the nitrogen is conserved: the molar flow falls by the oxygen taken |
| outlet ($x = L$) | none | diffusion along the channel is dropped, as in Ex09.2: the equation marches from the inlet |

Two simplifications carry over from Ex09.2, with numbers. The Mach number is below 5e-03, so the gas is incompressible and $c$ is constant. The velocity is Poiseuille's from the inlet; with Reynolds numbers of 7 to 22 the true entrance length is 2 to 4 mm (Ex09.2's low-Reynolds estimate), and the scoring starts at 5 mm. The suction at the wall is a thousandth of the axial speed, so it does not distort the parabola; it enters only through $v$.

## A4 · Why there is no exact solution

Ex09.2's channel has an exact solution because its flux is a constant and its flow does not change: beyond the entrance the field is linear along the channel plus a fixed quartic across it. Here the flux depends on the oxygen at the wall, the axial velocity depends on how much oxygen has been taken, the wall condition carries $1 - y$, and the equation gains the term $v\,\partial y/\partial z$ whose size depends on the local current. Each makes the problem nonlinear in $y$, the profile across the channel changes along it, and no closed form exists.

What remains exact are two **balances**: the nitrogen's, which gives $\sigma$ from the mean, and the oxygen's, which says how fast the mixing-cup mean falls. They need the wall value, so they cannot be solved alone; if the gas were mixed across the channel ($y_w = \bar y$, *plug flow*) they could, and the result is the hand calculation of section B2.

The reference model and the ground truth are therefore **numerical**: a Crank–Nicolson march along the channel (part C), checked on grids each twice as fine, by both balances, and at $B = 0$, $S = 0$, where it must reproduce Ex09.2's exact solution with the air's numbers.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the oxygen fraction at the electrode along the channel, beyond the first 5 mm | the test speeds 1.2, 1.9 and 2.7 m/s, which you do not train on | within **0.1 percentage point** of the ground truth everywhere |
| the Stefan term: does the velocity towards the electrode change the answer? | stoichiometry 5 (2.72 m/s) | answered **with a number**: the change of the oxygen at the electrode at the outlet, from your network trained with and without the term |
| the air speed for 15 % oxygen at the electrode at the outlet | found by a gradient through your network | within **1 %** of the ground truth's 1.6064 m/s |
| the time to answer a new speed online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, over the whole range of speeds. Its online answer is timed against the reference model, which must march the channel again for every new speed. The training time is reported apart, as the offline cost.

0.1 percentage point is Ex09.2's agreed accuracy. Here it is a real test: the plug-flow hand calculation misses the electrode by 0.18 point (part C), so a network has to resolve the profile across the channel to meet it.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network is a function of three inputs and returns one number:

$$(X,\ Z,\ s)\ \longmapsto\ \theta(X, Z; s)$$

with $X = x/L$, $Z = z/h$, $s = u_\mathrm{in}/(1\ \mathrm{m/s})$ the air speed at the inlet in m/s, and $\theta = (y - y_\mathrm{in})/Y$ the scaled change of the oxygen fraction, with Ex09.2's construction of the scale: $Y = N_0 L/(u_\mathrm{ref} h c)$ = 0.114 is the fall of the mean over the channel at 1 m/s with the inlet's current. The speed is an **input**: one training covers every speed from 1.0 to 3.0 m/s.

In [ ]:
# The scales, Ex09.2's construction with the air's numbers
print(f"X = x / L,  Z = z / h,  s = u_in / (1 m/s), {mp.S_MIN} to {mp.S_MAX}")
print(f"θ = (y - {mp.Y_IN}) / Y,   Y = N0 L / (u_ref h c) = {mp.Y:.3f}   (fuel side {ch.Y_SCALE:.3f})")
print(f"G = u_ref h² / (D L) = {mp.G:.4f}   (fuel side {ch.G:.4f}): the profile across the channel is of order G θ")
print(f"the wall's excess over the mean scales with Y G = N0 h / (D c) = {mp.Y*mp.G:.4f}   (fuel side {ch.Y_SCALE*ch.G:.4f})")

**What you should see.** $Y$ = 0.114, half the fuel side's (four electrons, not two); $G$ = 0.0526, four times the fuel side's (slower diffusion); their product, which sets the difference between the electrode and the mean, 0.0060 against 0.0029: about twice the fuel side's, as Ex09.2's review found.

## B2 · The physics the network is trained with

In the scaled variables, with $y_w = 0.21 + Y\theta(X, 0)$, $g = g(y_w)$ and $\sigma(X)$ the molar flow left,

$$6Z(1-Z)\,G\,s\,\sigma\,\frac{\partial \theta}{\partial X} - G\,Y\,g\,f(Z)\,\frac{\partial \theta}{\partial Z} = \frac{\partial^2 \theta}{\partial Z^2}$$

$$\theta = 0 \ \text{at}\ X = 0, \qquad
\frac{\partial \theta}{\partial Z} = G\,g\,(1 - y_w) \ \text{at}\ Z = 0, \qquad
\frac{\partial \theta}{\partial Z} = 0 \ \text{at}\ Z = 1, \qquad
\sigma = \frac{1 - 0.21}{1 - \bar y}$$

The second term on the left is the **Stefan term**, the cross velocity carrying oxygen towards the wall. Integrated over the height with the flow's weight, the equation gives the **balance of the oxygen's molar flow**:

$$s\,\sigma\,\frac{d\bar\theta}{dX} = -\,g(y_w)\,(1 - \bar y)$$

| loss term | where it is evaluated | what it enforces |
|---|---|---|
| $\mathcal{L}_\mathrm{PDE}$ | points inside $(X, Z, s)$ | the species equation with the falling flow and the Stefan term |
| $\mathcal{L}_\mathrm{inlet}$ | points on $X = 0$ | the air fed, 21 % oxygen |
| $\mathcal{L}_\mathrm{electrode}$ | points on $Z = 0$ | the nonlinear wall condition, $g$ and $1 - y_w$ at the network's own value |
| $\mathcal{L}_\mathrm{interconnect}$ | points on $Z = 1$ | the impermeable wall |
| $\mathcal{L}_\mathrm{balance}$ | quadrature columns at random $(X, s)$ | the oxygen balance, with $\sigma$ from the network's own mean |

Every residual is divided by $G$, as in Ex09.2. **One simplification in the equation's term**: $\sigma$ needs the flow-weighted mean, a quadrature column at every point. The given loss takes $\sigma$ from the hand calculation below instead, and holds the network's own mean by the balance term; part C shows that the ground truth march with that same simplification changes the electrode by about 1e-05 at most, a hundredth of the tolerance.

**What can be built in.** The **hand calculation** is the plug-flow mean: the gas mixed across the channel, the current at the mean's oxygen fraction, the molar flow falling,

$$s\,\sigma_p\,\frac{d\theta_p}{dX} = -\,g(y_p)\,(1 - y_p), \qquad \theta_p(0) = 0$$

which is separable and solved for any $X$ and $s$ by six Newton steps (`mp.plug_flow`, differentiable by autograd). Whether you build it in is your choice in part D.

In [ ]:
# The hand calculation against Ex09.2's balance with the air's numbers
X = np.linspace(0, 1, 401)
s5 = mp.speed_for_stoichiometry(5.0)
fig, ax = plt.subplots(figsize=(8, 3.4))
for s, c in ((1.2, "tab:red"), (1.9, "tab:blue"), (2.7, "tab:green")):
    ax.plot(100 * X, 100 * mp.oxygen(mp.plug_flow(X, s)), color=c, label=f"u = {s:.1f} m/s, plug flow, this project")
    ax.plot(100 * X, 100 * (mp.Y_IN - mp.Y * X / s), color=c, ls="--", lw=0.8)
ax.set_xlabel("x [mm]"); ax.set_ylabel("oxygen fraction, mixed across [%]"); ax.legend(fontsize=8)
ax.set_title("dashed: uniform current and constant molar flow (Ex09.2's balance)", fontsize=9)
plt.tight_layout(); plt.show()
yb = mp.Y_IN - mp.Y / s5
print(f"at stoichiometry 5 ({s5:.2f} m/s), oxygen at the outlet, mixed across:")
print(f"   uniform current, constant flow (the fuel side's balance): {100*yb:.2f} %")
print(f"   uniform current, flow falling, (y_in - Y)/(1 - Y):         {100*(mp.Y_IN - mp.Y/s5)/(1 - mp.Y/s5):.2f} %")
print(f"   current following the oxygen, flow falling (plug flow):    {100*mp.oxygen(mp.plug_flow(1.0, s5)):.2f} %")

**What you should see.** The oxygen falls along the channel more slowly than the straight dashed lines, for two reasons that the three numbers separate at stoichiometry 5: Ex09.2's constant-flow balance says 16.80 %; with the molar flow falling, 17.54 % (J. Khazaei's 0.74 point); with the current falling as the oxygen does, 17.60 %. The first correction is the one-sided exchange, the second the coupling.

## B3 · The training points

`mp92b.py` provides Ex09.2's two samplers on this range of speeds: `mp.sample_channel(n, seed)`, points $(X, Z, s)$ inside, a fifth of them in the first 5 % of the length; and `mp.sample_faces(n, seed)`, points on the inlet, the electrode and the interconnect. The speed is uniform from 1.0 to 3.0 m/s; the three **test speeds, 1.2, 1.9 and 2.7 m/s**, are not trained on on purpose.

In [ ]:
# The training points
pts = mp.sample_channel(4000)                            # (X, Z, s) inside
inlet, electrode_pts, interconnect = mp.sample_faces(400)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].scatter(100 * pts[:, 0], pts[:, 1], s=1, alpha=0.4, label="inside")
ax[0].scatter(100 * electrode_pts[:, 0], electrode_pts[:, 1], s=3, label="electrode")
ax[0].scatter(100 * interconnect[:, 0], interconnect[:, 1], s=3, label="interconnect")
ax[0].scatter(100 * inlet[:, 0], inlet[:, 1], s=3, label="inlet")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("Z = z/h"); ax[0].legend(fontsize=7, loc="center right")
ax[1].hist(pts[:, 2], bins=40)
for s in mp.S_TEST:
    ax[1].axvline(s, color="red", lw=1)
ax[1].set_xlabel("air speed at the inlet s [m/s]"); ax[1].set_ylabel("points")
plt.tight_layout(); plt.show()

**What you should see.** Left, the channel (in scaled height) with its points, denser in the first 5 mm; right, a flat distribution of speeds with the three test speeds in red.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the solver an engineer would run: the march on an engineering grid | the same scheme on a grid fine enough to be a thousand times more exact than anything scored |
| grid | 21 × 4 nodes (along × across) | 1601 × 81 nodes |
| its job | to be **timed** against the network, for a new speed and for the 15 % speed | to **score** the network and the reference |

## C2 · The scheme, step by step

1. **Nodes, conduction across, walls by ghost nodes** as in Ex09.2: central differences across the channel, the interconnect mirrored, the electrode's ghost carrying $G\,g\,(1 - y_w)$.
2. **Along the channel: Crank–Nicolson**, second order, one tridiagonal solve per step.
3. **The molar flow marched beside the field.** $\sigma$ falls by the oxygen the wall takes over the step, $d\sigma/dX = -Y g/s$, by the trapezoid rule; the axial velocity of the step uses $\sigma$ at its middle.
4. **The nonlinear terms by three iterations per step.** The electrode's flux is linearised on its node (Newton); $g$ in the Stefan term and $\sigma$ are taken at the last estimate. They change by parts in ten thousand within a step, and three iterations reach rounding.
5. **Rannacher's start**: the first two columns as two fully implicit half steps each, as Ex09.2 found necessary for a wall row with no marching term.
6. **The flow carried exactly**: the velocity row is divided by its trapezoid integral, $1 - h_Z^2$, so the grid carries the flow fed and the balances hold on any grid.

`mp.March(nx, nz).run(s)` returns $\theta$ (nz, nx) and $\sigma$ (nx); `mp.reference(s)` runs it on the engineering grid; `mp.speed_for_oxygen(0.15)` finds the speed by the secant method.

In [ ]:
# The ground truth: its convergence study at 1.9 m/s, from the truth file
G = np.load("MP9.2B_truth.npz")
print("grid (nx × nz)   oxygen at the electrode at the outlet   balance off by: oxygen   nitrogen")
for g, yw, bo, bn in zip(G["conv_grids"], G["conv_y_wall_outlet"], G["conv_balance_o2"], G["conv_balance_n2"]):
    print(f"{int(g[0]):5d} × {int(g[1]):3d}          {100*yw:.5f} %                        {abs(bo):.1e}   {abs(bn):.1e}")
d = np.abs(np.diff(G["conv_y_wall_outlet"]))
print("change of the outlet value between grids:", " ".join(f"{v:.1e}" for v in d), "; ratios", np.round(d[:-1] / d[1:], 1))
print(f"Richardson extrapolation: {100*float(G['conv_y_wall_outlet_extrapolated']):.5f} %")
print(f"at B = 0, S = 0, the march against Ex09.2's exact solution (air, reversed): {float(G['zero_strength_error']):.1e} at most beyond 5 mm")
print(f"σ from the hand calculation instead of the march's own balance changes the electrode by at most {mp.check_sigma_plug(1.2):.1e}")

**What you should see.** Five grids, each twice as fine. The change of the electrode's oxygen at the outlet falls by four each time (ratios 4.1, 4.0 and 4.0: second order); the ground truth's grid is within 2e-07 of the extrapolated value. Both balances close to 1e-08 on it. At $B = 0$, $S = 0$ the march reproduces Ex09.2's exact solution, with the air's numbers, to 8.5e-06 beyond 5 mm: that remainder is the entrance, which at 2.7 m/s and the air's four-times-larger $G$ reaches past 5 mm (beyond 10 mm it is 3.9e-07). And taking $\sigma$ from the hand calculation, as the given loss does, changes the electrode by 7.9e-06 at most at 1.2 m/s, the slowest test speed.

In [ ]:
# The Stefan question, from the ground truth at stoichiometry 5
full, no_cross, none = G["stefan_stoich5"]
print(f"oxygen at the electrode at the outlet, stoichiometry 5 ({float(G['speed_stoich5']):.2f} m/s):")
print(f"   the whole one-sided exchange:                         {100*full:.3f} %")
print(f"   without the Stefan term v ∂y/∂z in the equation:      {100*no_cross:.3f} %   ({100*(full - no_cross):+.4f} points)")
print(f"   neither (Ex09.2's equimolar model, air's numbers):    {100*none:.3f} %   ({100*(full - none):+.3f} points)")

**What you should see.** 17.427 % with everything; 17.420 % without the Stefan term in the equation, a change of 0.0076 percentage point, a thirteenth of the 0.1 tolerance; and 16.686 % with Ex09.2's equimolar model, 0.74 point off, seven times the tolerance. **The one-sided exchange matters through the falling molar flow and the $1 - y_w$ at the wall, not through the cross velocity in the equation.** Your network will answer the same question in part E.

## C3 · The reference model: accuracy and time

The reference grid was chosen as the coarsest that meets 0.1 percentage point at every speed and the 15 % speed within 1 %: 21 × 3 is 0.062 point off but its speed 1.0 % off; 21 × 4 is 0.022 point and 0.35 %. It runs on the CPU: a march of 21 small tridiagonal solves gains nothing from a GPU.

In [ ]:
# The reference model against the ground truth, at the three test speeds
worst = 0.0
for s in mp.S_TEST:
    r = mp.reference(s)
    truth = np.interp(r["X"], G["X_wall"], G[f"y_wall_{s:.1f}"])         # the ground truth at the reference's nodes
    e = np.abs(r["y_wall"] - truth)[r["X"] >= mp.X_SCORE].max()
    worst = max(worst, e)
    print(f"u = {s:.1f} m/s: oxygen at the electrode at the outlet {100*r['y_wall'][-1]:.2f} % (truth {100*G[f'y_wall_{s:.1f}'][-1]:.2f} %), "
          f"worst error {100*e:.3f} points; molar flow left {100*r['sigma'][-1]:.2f} %")
t_ref = middle(lambda: mp.reference(2.0))                # the online cost of a new speed
s15, n_march, _ = mp.speed_for_oxygen(0.15)
t_design = middle(lambda: mp.speed_for_oxygen(0.15), reps=3)   # the online cost of the design question
print(f"\nreference model: worst {100*worst:.3f} percentage points, {t_ref*1e3:.2f} ms per new speed")
print(f"speed for 15 % at the electrode at the outlet: {s15:.4f} m/s (truth {float(G['speed_15']):.4f}, "
      f"{100*(s15/float(G['speed_15'])-1):+.2f} %), {n_march} marches, {t_design*1e3:.1f} ms")
REF = {"t_ref_s": t_ref, "worst_ref": worst, "t_design_s": t_design, "speed_15_ref": s15}   # kept for part E

**What you should see.** At every test speed the reference model is within 0.022 percentage point at the electrode, and the molar flow at the outlet is 90.9 to 95.9 % of the inlet's. One march takes several milliseconds (6.6 to 16.5 ms on the authors' CPU, shared with other jobs); the 15 % speed, 1.6121 m/s, is 0.35 % from the ground truth's 1.6064 m/s, in eight marches and 50 to 110 ms. Its times are kept in `REF` for part E.

In [ ]:
# The ground truth at the three test speeds
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for s in mp.S_TEST:
    k = f"{s:.1f}"
    ax[0].plot(100 * G["X_wall"], 100 * G[f"y_wall_{k}"], label=f"u = {s:.1f} m/s, electrode")
    ax[0].plot(100 * G["X_wall"], 100 * mp.oxygen(mp.plug_flow(G["X_wall"], s)), "k:", lw=0.8)
    ax[1].plot(100 * (G[f"field_{k}"][:, -1] - G[f"y_mean_{k}"][-1]), G["Z"], label=f"u = {s:.1f} m/s")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("oxygen fraction [%]"); ax[0].legend(fontsize=8)
ax[0].set_title("dotted: the hand calculation (plug flow)", fontsize=9)
ax[1].set_xlabel("oxygen at the outlet minus the mixing-cup mean [% points]"); ax[1].set_ylabel("Z = z/h"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
for s in mp.S_TEST:
    e = np.abs(G[f"y_wall_{s:.1f}"] - mp.oxygen(mp.plug_flow(G["X_wall"], s)))[G["X_wall"] >= mp.X_SCORE].max()
    print(f"u = {s:.1f} m/s: the hand calculation misses the electrode by up to {100*e:.3f} percentage points")

**What you should see.** Left, the oxygen along the electrode falling from 21 % to 12.96, 15.92 and 17.41 % at the outlet, a little below the hand calculation (dotted). Right, the profile across the channel at the outlet: lowest at the electrode, about 0.18 point below the mean. The hand calculation misses the electrode by up to 0.176 percentage points at every speed, more than the 0.1 asked: the network has to resolve the profile across the channel.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given.

**The contract.** Your network is a PyTorch module called `net`, used as `net(X, Z, s)`:

| argument | meaning | shape |
|---|---|---|
| `X`, `Z` | $x/L$ and $z/h$, in 0 to 1 | `(n, 1)` |
| `s` | the air speed at the inlet in m/s, 1.0 to 3.0 | `(n, 1)` |
| returns | the scaled field $\theta = (y - 0.21)/Y$, with $Y$ = 0.114 | `(n, 1)` |

## D2 · The loss, given

`losses(net, seed, stefan=True)` draws training points with the samplers of part B (the same points for the same seed) and returns the four residual terms of section B2's table, each divided by $G$; `stefan=False` drops the Stefan term from the equation, which question 6 needs. `balance_loss(net, seed)` is the oxygen balance at 300 random $(X, s)$, the mean by 16-point Gauss–Legendre quadrature with the flow's weight and its slope by autograd, $\sigma$ from the network's own mean. Two details:

* **$\sigma$ in the equation's term comes from the hand calculation** (`mp.sigma_plug`), which saves a quadrature column at every point; part C showed it costs about 1e-05 at most.
* **The Stefan term needs the wall value at the same $X$**: the loss evaluates the network a second time at $Z = 0$ for every interior point.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a, grad=False):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1).requires_grad_(grad)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

def g_torch(theta_w):
    """g = N/N0 at the electrode's scaled oxygen fraction θ_w, in torch (mp.g_flux in NumPy)."""
    y = torch.clamp(mp.Y_IN + mp.Y * theta_w, min=1e-6)                  # the oxygen fraction at the wall, kept positive
    return (1 - mp.B_COUPLE) + mp.B_COUPLE * (y / (y + mp.K_O2)) / (mp.Y_IN / (mp.Y_IN + mp.K_O2))

def losses(net, seed=88, n_in=4000, n_face=400, stefan=True):
    # inside: 6 Z (1 - Z) G s σ θ_X - G Y g(θ_w) f(Z) θ_Z = θ_ZZ, divided by G
    p = mp.sample_channel(n_in, seed)
    X, Z, s = T(p[:, 0], True), T(p[:, 1], True), T(p[:, 2])
    th = net(X, Z, s)
    th_w = net(X, torch.zeros_like(Z), s)                # the electrode's value at the same X and speed
    sigma = mp.sigma_plug(X, s, lib=torch)               # the molar flow left, from the hand calculation's mean
    cross = mp.G * mp.Y * g_torch(th_w) * mp.f_cross(Z) * d(th, Z) if stefan else 0.0   # the Stefan velocity's term
    r_pde = (d(d(th, Z), Z) - 6 * Z * (1 - Z) * mp.G * s * sigma * d(th, X) + cross) / mp.G
    inlet, el, ic = mp.sample_faces(n_face, seed)
    # the inlet, X = 0: θ = 0 (21 % oxygen), divided by G
    r_in = net(T(inlet[:, 0]), T(inlet[:, 1]), T(inlet[:, 2])) / mp.G
    # the electrode, Z = 0: θ_Z = G g(θ) (1 - y_w), divided by G
    Ze = T(el[:, 1], True)
    the = net(T(el[:, 0]), Ze, T(el[:, 2]))
    r_el = d(the, Ze) / mp.G - g_torch(the) * (1 - (mp.Y_IN + mp.Y * the))
    # the interconnect, Z = 1: θ_Z = 0, divided by G
    Zc = T(ic[:, 1], True)
    r_ic = d(net(T(ic[:, 0]), Zc, T(ic[:, 2])), Zc) / mp.G
    return {"pde": r_pde.pow(2).mean(), "inlet": r_in.pow(2).mean(),
            "electrode": r_el.pow(2).mean(), "interconnect": r_ic.pow(2).mean()}

# the balance of the oxygen's molar flow (section B2): s σ dθ̄/dX = -g(θ_w) (1 - ȳ), divided by G
_zq, _wq = np.polynomial.legendre.leggauss(16)            # Gauss-Legendre nodes and weights on [-1, 1]
ZQ = T((_zq + 1) / 2)                                    # moved to [0, 1]
WQ = T(_wq / 2 * 6 * ((_zq + 1) / 2) * (1 - (_zq + 1) / 2)).ravel()   # the weights carry the flow's 6 Z (1 - Z); they sum to 1

def balance_loss(net, seed=7, n=300):
    q = mp.sample_channel(n, seed)[:, [0, 2]]             # (X, s) pairs
    X = T(np.repeat(q[:, 0], 16), True)                  # each pair once per quadrature node
    s = T(np.repeat(q[:, 1], 16))
    th = net(X, ZQ.repeat(n, 1), s).reshape(n, 16)
    mean = (th * WQ).sum(1, keepdim=True)                 # θ̄ at each (X, s)
    rate = (d(th.reshape(-1, 1), X).reshape(n, 16) * WQ).sum(1, keepdim=True)   # dθ̄/dX, by autograd
    Xw, sw = T(q[:, 0]), T(q[:, 1])
    wall = net(Xw, torch.zeros_like(Xw), sw)              # θ at the electrode
    sigma = (1 - mp.Y_IN) / (1 - (mp.Y_IN + mp.Y * mean))  # the molar flow left, from the network's own mean
    return ((sw * sigma * rate + g_torch(wall) * (1 - (mp.Y_IN + mp.Y * mean))) / mp.G).pow(2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide what the network sees (the three inputs scaled to order one, or features), what is built in (the hand calculation `mp.plug_flow(X, s, lib=torch)` is differentiable; the inlet; the scale $G$ of the profile across), and its size. The field is the plug-flow mean, which falls by up to 9 percentage points along the channel, plus a profile across of about 0.2 point.

In [ ]:
# TODO 1 - your network. net(X, Z, s) -> θ, every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, X, Z, s):
        ...                                              # return θ, shape (n, 1)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses` and `balance_loss`. Decide the optimiser or optimisers, the steps, the learning rate, the weights of the terms and the points. Keep the history of each term for TODO 3 and set `TRAIN_SECONDS`, the offline cost. `quick_score()` below gives the worst error of the oxygen fraction at the electrode at the three test speeds, beyond 5 mm, in percentage points.

In [ ]:
# A quick score, given: the worst error at the electrode at the test speeds, percentage points
G = np.load("MP9.2B_truth.npz")
def quick_score():
    Xw = G["X_wall"]; keep = Xw >= mp.X_SCORE; worst = 0.0
    for s in mp.S_TEST:
        with torch.no_grad():
            th = net(T(Xw), T(np.zeros_like(Xw)), T(np.full_like(Xw, s))).cpu().numpy().ravel()
        worst = max(worst, np.abs(mp.oxygen(th) - G[f"y_wall_{s:.1f}"])[keep].max())
    return 100 * worst

In [ ]:
# TODO 2 - your training
history = {"pde": [], "inlet": [], "electrode": [], "interconnect": [], "balance": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst error at the electrode {quick_score():.3f} percentage points")

## D5 · TODO 3 · Your figures

Draw at least the loss terms during training, and the oxygen along the electrode at one test speed against the ground truth, with its error. Save each figure you want in the report as `MP9.2B_report_<name>.png` (in `cc.output_path`).

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in `R`:

* the oxygen fraction along the electrode (401 points) at the test speeds 1.2, 1.9 and 2.7 m/s, at 0.8 and 3.3 m/s outside the range, and at the outlet for eleven speeds from 1.0 to 3.0 m/s;
* the field at 1.9 m/s on the ground truth's 401 × 41 nodes, and the outlet at stoichiometry 5;
* **the air speed for 15 % oxygen at the electrode at the outlet by Newton's method through the network**, the derivative by autograd, started from Ex09.2's balance, $Y/(0.21 - 0.15)$;
* the **online time** of a new speed - the oxygen along the electrode on 101 points from one call of `net` - and of 10 and 100 speeds at once, against the reference model run 1, 10 and 100 times; and the Newton search against the reference model's secant search.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
Xe = np.linspace(0.0, 1.0, 401)                          # along the electrode
def electrode_profile(speeds, X=Xe):
    # the oxygen fraction at the electrode, one row per speed, from one call of the network
    speeds = np.atleast_1d(speeds); n = len(X)
    with torch.no_grad():
        th = net(T(np.tile(X, len(speeds))), T(np.zeros(n * len(speeds))), T(np.repeat(speeds, n)))
    return mp.oxygen(th.reshape(len(speeds), n).cpu().numpy())

speeds = {"test": np.array(mp.S_TEST), "outside": np.array([0.8, 3.3]), "sweep": np.linspace(1.0, 3.0, 11)}
out = {f"y_wall_{k}": electrode_profile(v) for k, v in speeds.items()}
out.update({f"speeds_{k}": v for k, v in speeds.items()}, X=Xe)
out["outlet_stoich5"] = electrode_profile([float(G["speed_stoich5"])])[0, -1]

# the field at 1.9 m/s on the ground truth's nodes
XX, ZZ = np.meshgrid(G["X"], G["Z"])
with torch.no_grad():
    out["field_1.9"] = mp.oxygen(net(T(XX), T(ZZ), T(np.full(XX.size, 1.9))).reshape(XX.shape).cpu().numpy())

# the air speed for 15 % oxygen at the electrode at the outlet, by Newton's method through the network
def design(target=0.15, steps=5):
    s = torch.tensor([[mp.Y / (mp.Y_IN - target)]], device=DEVICE, requires_grad=True)    # Ex09.2's balance, the start
    one, zero = torch.ones(1, 1, device=DEVICE), torch.zeros(1, 1, device=DEVICE)
    for _ in range(steps):
        f = mp.Y_IN + mp.Y * net(one, zero, s) - target                                     # oxygen at the outlet's electrode minus the target
        df = torch.autograd.grad(f.sum(), s)[0]                                             # its derivative in s, by autograd
        with torch.no_grad():
            s -= f / df                                                                     # Newton's step
    return float(s.detach())
out["speed_15"] = np.array(design())

# online time: the network against the reference model, on this runtime
timing = {}
for k in (1, 10, 100):
    ps = np.linspace(1.05, 2.95, k)
    timing[f"net_{k}"] = middle(lambda: electrode_profile(ps, np.linspace(0, 1, 101)))
    timing[f"ref_{k}"] = middle(lambda: [mp.reference(p) for p in ps], reps=3 if k == 100 else 5)
timing["design_net"] = middle(design)
timing["design_ref"] = REF["t_design_s"]
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP9.2B_net.pt"))   # the trained weights, beside the report
for k in (1, 10, 100):
    print(f"online, {k:3d} new speed(s): network {timing[f'net_{k}']*1e3:7.2f} ms, reference {timing[f'ref_{k}']*1e3:7.2f} ms "
          f"({timing[f'ref_{k}']/timing[f'net_{k}']:.1f} times)")
print(f"speed for 15 %: network {float(out['speed_15']):.4f} m/s in {timing['design_net']*1e3:.1f} ms, "
      f"reference {REF['speed_15_ref']:.4f} m/s in {timing['design_ref']*1e3:.1f} ms; truth {float(G['speed_15']):.4f} m/s")
print(f"oxygen at the electrode at the outlet, stoichiometry 5: network {100*float(out['outlet_stoich5']):.3f} %, "
      f"truth {100*float(G['stefan_stoich5'][0]):.3f} %")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's times `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP9.2B_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): worst {100*REF['worst_ref']:.3f} percentage points, {REF['t_ref_s']*1e3:.2f} ms per new speed")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in `METRICS` (a name and a number each): the worst error at the electrode beyond 5 mm at each test speed (percentage points); the error of the 15 % speed (per cent); the largest error of the field at 1.9 m/s; the oxygen at the electrode at the outlet at stoichiometry 5 against the ground truth; the online times of a new speed and of the speed search, network and reference, and their ratios; the offline training time. Then say in one line each whether the requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>55}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP9.2B_report_<name>.png`, the figures your answers refer to. At least: the error along the electrode at the three test speeds; the error of the field at 1.9 m/s; the oxygen at the outlet's electrode against the speed (`R["y_wall_sweep"]`) with the 15 % speed found; and the online times for 1, 10 and 100 speeds.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 5 are about your network and its result, 6 to 9 ask you to use it, 10 concludes. The next cell writes `MP9.2B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features or hand calculation built in), layers, neurons and number of weights. Why this design?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training.": """
""",
    "3. Your training: optimiser(s), steps, learning rates, number of points of each kind, and the offline time on the T4 in seconds.": """
""",
    "4. The oxygen fraction at the electrode at 1.2, 1.9 and 2.7 m/s: give the worst error at each speed in percentage points. Are all within 0.1? Where along the channel is your error largest, and why there?": """
""",
    "5. The online time to answer a new speed: your network against the reference model on the same runtime, in milliseconds, and the ratio; the same for 10 and 100 speeds, and for the search for the 15 % speed. What does each method spend its time on?": """
""",
    "6. The Stefan question: train your network again with losses(..., stefan=False). By how many percentage points does the oxygen at the electrode at the outlet at stoichiometry 5 change? Compare with the ground truth's 0.0076 and with the 0.74 point of Ex09.2's equimolar model, and say what in the one-sided exchange matters.": """
""",
    "7. The air speed for 15 % oxygen at the electrode at the outlet: your network's value, the ground truth's, the error in per cent, and the Newton steps. What stoichiometry is it, and how much slower than Ex09.2's balance's 1.90 m/s?": """
""",
    "8. At stoichiometry 5, how far does the current density fall along the electrode, and by how much has the molar flow fallen at the outlet? By how many percentage points does the electrode lie below the mixing-cup mean there?": """
""",
    "9. Outside the training range and with another seed: your errors at 0.8 and 3.3 m/s, and the worst error and 15 % speed of a second training from another seed. How robust is your result?": """
""",
    "10. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP9.2B — The Air Channel", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP9.2B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP9.2B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP9.2B_report.md into MP9.2B_report.pdf, with any figure
# saved as MP9.2B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP9.2B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP9.2B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP9.2B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP9.2B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass